# NB_Setup_Folders_And_Tables
Creates directories and initializes metadata tables in LH_ReferenceData.

In [ ]:
# 1. Create Folder Structures in Files
lakehouses = ["LH_Claims", "LH_Members", "LH_Providers"]
for lh in lakehouses:
    # Note: mssparkutils.fs depends on having the lakehouse attached
    # This logic assumes folders are being created in the current default lakehouse context
    mssparkutils.fs.mkdirs("Files/dev/bronze")
    mssparkutils.fs.mkdirs("Files/dev/silver")
    mssparkutils.fs.mkdirs("Files/dev/gold")
    mssparkutils.fs.mkdirs("Files/raw")
    print(f"Folders initialized for default lakehouse.")

# 2. Initialize Reference Tables (if attached to LH_ReferenceData)
try:
    spark.sql("""
    CREATE TABLE IF NOT EXISTS config_environment (
        environment STRING, storage_path STRING, notification_email STRING
    ) USING DELTA
    """)
    
    spark.sql("""
    CREATE TABLE IF NOT EXISTS config_ingestion_metadata (
        source_id STRING, domain STRING, source_object STRING, 
        target_lakehouse STRING, target_table STRING, file_format STRING, 
        load_type STRING, watermark_column STRING, primary_key STRING, 
        active_flag BOOLEAN, created_date TIMESTAMP
    ) USING DELTA
    """)
    
    spark.sql("""
    CREATE TABLE IF NOT EXISTS platform_error_log (
        layer STRING, table_name STRING, error_message STRING, error_timestamp TIMESTAMP
    ) USING DELTA
    """)
    
    # Insert Sample Metadata for Claims if not present
    count = spark.sql("SELECT count(*) as count FROM config_ingestion_metadata").collect()[0]['count']
    if count == 0:
        spark.sql("""
        INSERT INTO config_ingestion_metadata VALUES 
        ('CL001', 'Claims', 'claims_sample.csv', 'LH_Claims', 'claims_bronze', 'csv', 'full', 'none', 'claim_id', true, current_timestamp())
        """)
    
    print("Metadata tables initialized in LH_ReferenceData.")
except Exception as e:
    print(f"Metadata setup skipped or failed: {str(e)}")